In [1]:
import pandas as pd
import numpy as np
import re

df = pd.read_excel('iqoo.xlsx')

df['MRP'] = df['MRP'].str.replace('₹', '').str.replace(' ', '').str.replace(',', '').astype(float)
df['Current Price'] = df['Current Price'].str.replace('₹', '').str.replace(',', '').astype(float)

def extract_discount(value):
    if pd.isna(value):
        return 0  # or return None
    return int(value.replace('% off', ''))
df['Discount_Number'] = df['Discount'].apply(extract_discount)

df.drop("Discount", axis=1, inplace=True)

brands = {
    'Apple': ['apple', 'iphone', 'ipad'],
    'motorola': ['motorola', 'moto'],
    'AI_plus': ['ai+', 'ai', 'aiplus', 'ai plus', 'a i'], 
    'IQOO': ['iqoo'],
    'Tecno': ['tecno'],
    'lava': ['lava'],
    'readmi': ['readmi', 'redmi'],  
    'realme': ['realme'],
    'POCO': ['poco'],
    'nothing': ['nothing'],
    'Infix': ['infix'],
    'Oppo': ['oppo'],
    'vivo': ['vivo'],
    'Google': ['google', 'pixel'],
    'OnePlus': ['oneplus', 'one plus'],
    'Samsung': ['samsung', 'galaxy'],
}

def get_brand(product_name):
    if pd.isna(product_name):
        return 'Unknown'
    
    product_name = str(product_name).lower()
    
    for brand, keywords in brands.items():
        for keyword in keywords:
            if keyword in product_name:
                return brand
    return 'Unknown'
df['Brand'] = df['Product Name'].apply(get_brand)

df['RAM'] = df['RAM'].str.extract(r'(\d+)').astype(int)


df['Storage'] = df['Storage'].str.extract(r'(\d+)').astype(int)


def extract_ratings_reviews(text):
    """
    Extract ratings and reviews from text
    Example: "2,46,375 Ratings & 9,265 Reviews"
    Returns: (ratings, reviews)
    """
    if pd.isna(text):
        return None, None
    
    text = str(text)
    
    # Extract ratings (numbers before 'Ratings')
    ratings_match = re.search(r'([\d,]+)\s*Ratings', text, re.IGNORECASE)
    ratings = ratings_match.group(1).replace(',', '') if ratings_match else None
    
    # Extract reviews (numbers before 'Reviews')
    reviews_match = re.search(r'([\d,]+)\s*Reviews', text, re.IGNORECASE)
    reviews = reviews_match.group(1).replace(',', '') if reviews_match else None
    
    return ratings, reviews

# Apply to dataframe
df[['Ratings', 'Reviews']] = df['Review text'].apply(
    lambda x: pd.Series(extract_ratings_reviews(x))
)

# Convert to numbers
df['Ratings'] = pd.to_numeric(df['Ratings'], errors='coerce')
df['Reviews'] = pd.to_numeric(df['Reviews'], errors='coerce')


def extract_battery(text):
    if pd.isna(text):
        return 0
    match = re.search(r'(\d+)', str(text))
    return int(match.group()) if match else 0

df['Battery'] = df['Battery'].apply(extract_battery)

In [2]:
df.head()

,Product Name,Current Price,MRP,Rating,Review text,RAM,Storage,Display,Camera,Battery,Processor,Product URL,Image URL,Discount_Number,Brand,Ratings,Reviews
0,"IQOO Z10R 5G (Aquamarine, 128 GB)",22999.0,23499.0,4.4,"5,660 Ratings & 329 Reviews",8,128,17.2 cm (6.77 inch) Display,50MP Rear Camera,5700,NaN,https://www.flipkart.com/iqoo-z10r-5g-aquamari...,https://rukminim2.flixcart.com/image/312/312/x...,2,IQOO,5660,329
1,"IQOO Z10X 5G (Titanium, 128 GB)",22997.0,22999.0,4.4,"10,960 Ratings & 558 Reviews",6,128,17.02 cm (6.7 inch) Display,50MP Rear Camera,6500,NaN,https://www.flipkart.com/iqoo-z10x-5g-titanium...,https://rukminim2.flixcart.com/image/312/312/x...,0,IQOO,10960,558
2,"IQOO Z10X 5G (Ultramarine, 128 GB)",22998.0,22999.0,4.4,"10,960 Ratings & 558 Reviews",6,128,17.02 cm (6.7 inch) Display,50MP Rear Camera,6500,NaN,https://www.flipkart.com/iqoo-z10x-5g-ultramar...,https://rukminim2.flixcart.com/image/312/312/x...,0,IQOO,10960,558
3,"IQOO 15 5G (Alpha, 512 GB)",78200.0,83999.0,4.6,111 Ratings & 14 Reviews,16,512,17.4 cm (6.85 inch) Display,50MP Rear Camera,7000,NaN,https://www.flipkart.com/iqoo-15-5g-alpha-512-...,https://rukminim2.flixcart.com/image/312/312/x...,6,IQOO,111,14
4,"IQOO 15 5G (Legend, 512 GB)",79900.0,83999.0,4.6,111 Ratings & 14 Reviews,16,512,17.4 cm (6.85 inch) Display,50MP Rear Camera,7000,NaN,https://www.flipkart.com/iqoo-15-5g-legend-512...,https://rukminim2.flixcart.com/image/312/312/x...,4,IQOO,111,14


In [3]:
df.isnull().sum()

Product Name         0
Current Price        2
MRP                 32
Rating               0
Review text          0
RAM                  0
Storage              0
Display              0
Camera               0
Battery              0
Processor          124
Product URL          0
Image URL            0
Discount_Number      0
Brand                0
Ratings              0
Reviews              0
dtype: int64

In [4]:
df['MRP'] = np.where(
    (df['Discount_Number'] == 0) & (df['MRP'].isna() | (df['MRP'] == '')),
    df['Current Price'],
    df['MRP']
)

In [5]:
df.isnull().sum()

Product Name         0
Current Price        2
MRP                  2
Rating               0
Review text          0
RAM                  0
Storage              0
Display              0
Camera               0
Battery              0
Processor          124
Product URL          0
Image URL            0
Discount_Number      0
Brand                0
Ratings              0
Reviews              0
dtype: int64

In [6]:
def get_iqoo_processor(product_name):
    if pd.isna(product_name):
        return 'Unknown Processor'
    
    # iQOO 13 Series
    if 'iQOO 13' in product_name:
        return 'Snapdragon 8 Gen 4'
    
    # iQOO 12 Series
    elif 'iQOO 12' in product_name:
        if 'Pro' in product_name:
            return 'Snapdragon 8 Gen 3'
        else:
            return 'Snapdragon 8 Gen 3'
    
    # iQOO 11 Series
    elif 'iQOO 11' in product_name:
        if 'Pro' in product_name:
            return 'Snapdragon 8 Gen 2'
        else:
            return 'Snapdragon 8 Gen 2'
    
    # iQOO 10 Series
    elif 'iQOO 10' in product_name:
        return 'Snapdragon 8+ Gen 1'
    
    # iQOO 9 Series
    elif 'iQOO 9' in product_name:
        return 'Snapdragon 888+'
    
    # iQOO 8 Series
    elif 'iQOO 8' in product_name:
        return 'Snapdragon 888'
    
    # iQOO 7 Series
    elif 'iQOO 7' in product_name:
        return 'Snapdragon 870'
    
    # iQOO Z Series
    elif 'iQOO Z9s' in product_name:
        return 'MediaTek Dimensity 7300'
    elif 'iQOO Z9' in product_name:
        if 'Pro' in product_name:
            return 'MediaTek Dimensity 8300'
        else:
            return 'MediaTek Dimensity 7200'
    elif 'iQOO Z8' in product_name:
        return 'MediaTek Dimensity 8200'
    elif 'iQOO Z7' in product_name:
        if 'Pro' in product_name:
            return 'MediaTek Dimensity 7200'
        else:
            return 'Snapdragon 782G'
    elif 'iQOO Z6' in product_name:
        return 'Snapdragon 778G+'
    
    # iQOO Neo Series
    elif 'iQOO Neo 10' in product_name:
        return 'Snapdragon 8 Gen 3'
    elif 'iQOO Neo 9' in product_name:
        if 'Pro' in product_name:
            return 'Snapdragon 8 Gen 2'
        else:
            return 'Snapdragon 8+ Gen 1'
    elif 'iQOO Neo 8' in product_name:
        return 'Snapdragon 8+ Gen 1'
    elif 'iQOO Neo 7' in product_name:
        return 'MediaTek Dimensity 8200'
    elif 'iQOO Neo 6' in product_name:
        return 'Snapdragon 870'
    
    else:
        return 'Unknown Processor'

df.loc[df['Processor'].isnull(), 'Processor'] = df.loc[df['Processor'].isnull(), 'Product Name'].apply(get_iqoo_processor)

In [7]:
df.isnull().sum()

Product Name       0
Current Price      2
MRP                2
Rating             0
Review text        0
RAM                0
Storage            0
Display            0
Camera             0
Battery            0
Processor          0
Product URL        0
Image URL          0
Discount_Number    0
Brand              0
Ratings            0
Reviews            0
dtype: int64

In [8]:
df['Current Price'] = df['Current Price'].fillna(df['Current Price'].median())
df['MRP'] = df['MRP'].fillna(df['Current Price'])

In [9]:
df.isnull().sum()

Product Name       0
Current Price      0
MRP                0
Rating             0
Review text        0
RAM                0
Storage            0
Display            0
Camera             0
Battery            0
Processor          0
Product URL        0
Image URL          0
Discount_Number    0
Brand              0
Ratings            0
Reviews            0
dtype: int64

In [10]:
df.to_excel('Cleaned_iqoo.xlsx', index=False)